In [1]:
#Cell 1 – Chia 80/20 train/val
from pathlib import Path
import random
import shutil

# === CHỈNH LẠI ĐƯỜNG DẪN NÀY CHO ĐÚNG MÁY ÔNG ===
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")

IMG_ROOT = ROOT / "images"
TRAIN_DIR = IMG_ROOT / "train"
VAL_DIR   = IMG_ROOT / "val"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

# Lấy ảnh NẰM TRỰC TIẾP trong images/ (chưa chia)
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
all_imgs = [
    p for p in IMG_ROOT.iterdir()
    if p.is_file() and p.suffix.lower() in exts
]

if not all_imgs:
    print("Không thấy ảnh trực tiếp trong 'images/' – có thể ông đã chia train/val rồi.")
else:
    print("Tổng ảnh ban đầu:", len(all_imgs))

    random.seed(1337)
    random.shuffle(all_imgs)

    split_idx = int(len(all_imgs) * 0.8)
    train_imgs = all_imgs[:split_idx]
    val_imgs   = all_imgs[split_idx:]

    def move_batch(files, dst_dir):
        for p in files:
            dst = dst_dir / p.name
            print("MOVE:", p, "->", dst)
            shutil.move(str(p), str(dst))

    move_batch(train_imgs, TRAIN_DIR)
    move_batch(val_imgs, VAL_DIR)

    print("Done. Train:", len(list(TRAIN_DIR.glob('*'))),
          "| Val:", len(list(VAL_DIR.glob('*'))))


Không thấy ảnh trực tiếp trong 'images/' – có thể ông đã chia train/val rồi.


In [2]:

from ultralytics import YOLO
from pathlib import Path

# === ĐƯỜNG DẪN GIỐNG CELL 1 ===
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")
IMG_TRAIN = ROOT / "images" / "train"
IMG_VAL   = ROOT / "images" / "val"

LBL_TRAIN = ROOT / "labels" / "train"
LBL_VAL   = ROOT / "labels" / "val"

LBL_TRAIN.mkdir(parents=True, exist_ok=True)
LBL_VAL.mkdir(parents=True, exist_ok=True)

# === CHỌN MODEL YOLO PRETRAIN (COCO) ===
# nếu ông có sẵn yolo11n.pt trong project thì chỉnh path cho đúng
yolo_model = YOLO("yolo11n.pt")   # hoặc "yolo11s.pt"
print("Loaded YOLO:", yolo_model.model.__class__.__name__)

# Chỉ giữ các class trông giống đồ ăn / chén / dĩa / ly...
ALLOWED = {
    "bowl", "cup", "wine glass", "bottle",
    "fork", "knife", "spoon",
    "pizza", "cake", "sandwich", "hot dog",
    "dining table"
}

exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

def auto_label_split(split_name, img_dir: Path, lbl_dir: Path):
    img_files = []
    for e in exts:
        img_files.extend(img_dir.rglob(e))

    print(f"\n[{split_name}] Số ảnh:", len(img_files))

    for img_path in sorted(img_files):
        label_path = lbl_dir / (img_path.stem + ".txt")

        # Nếu đã có label rồi thì bỏ qua (cho dễ rerun)
        if label_path.exists():
            # print("SKIP (đã có label):", img_path.name)
            continue

        results = yolo_model.predict(
            source=str(img_path),
            conf=0.3,
            verbose=False,
        )

        lines = []

        for r in results:
            if r.boxes is None:
                continue

            h, w = r.orig_img.shape[:2]

            for b in r.boxes:
                cls_idx = int(b.cls.item())
                cls_name = r.names[cls_idx]

                # Lọc class COCO
                if ALLOWED and cls_name not in ALLOWED:
                    continue

                x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().tolist()

                # đổi sang xywh normalized
                xc = (x1 + x2) / 2.0 / w
                yc = (y1 + y2) / 2.0 / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h

                # class duy nhất: 0 (dish)
                lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        if lines:
            label_path.write_text("\n".join(lines), encoding="utf-8")
        else:
            # không detect gì – vẫn ghi file rỗng để dễ kiểm tra
            label_path.write_text("", encoding="utf-8")

        print(f"{img_path.name:30s} -> {label_path.name} ({len(lines)} box)")

# Chạy auto-label cho train & val
auto_label_split("train", IMG_TRAIN, LBL_TRAIN)
auto_label_split("val",   IMG_VAL,   LBL_VAL)

print("\n✅ Xong auto-label. Cấu trúc hiện tại:")
print(" -", IMG_TRAIN)
print(" -", IMG_VAL)
print(" -", LBL_TRAIN)
print(" -", LBL_VAL)


Loaded YOLO: DetectionModel

[train] Số ảnh: 302

[val] Số ảnh: 76

✅ Xong auto-label. Cấu trúc hiện tại:
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/images/train
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/images/val
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/labels/train
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/labels/val


In [3]:
from pathlib import Path

# ĐƯỜNG DẪN GIỐNG NHƯ CÁC CELL TRƯỚC
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")

yaml_path = ROOT / "data_food_plate.yaml"

yaml_text = f"""path: {ROOT}
train: images/train
val: images/val

names:
  0: dish
"""

yaml_path.write_text(yaml_text, encoding="utf-8")
print("Đã tạo file cấu hình:", yaml_path)
print("Nội dung:")
print("--------------------------------")
print(yaml_text)
print("--------------------------------")


Đã tạo file cấu hình: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/data_food_plate.yaml
Nội dung:
--------------------------------
path: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables
train: images/train
val: images/val

names:
  0: dish

--------------------------------


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Thư mục đang chứa file yolo.ipynb
NB_DIR = Path.cwd()                     # .../Jupyter/evaluate
print("NB_DIR:", NB_DIR)

# Thư mục để YOLO lưu run (để gọn cứ để ngay trong evaluate luôn)
YOLO_PROJECT = NB_DIR                   # hoặc NB_DIR / "runs_yolo"

ROOT_FOODTABLES = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")
DATA_YAML = ROOT_FOODTABLES / "data_food_plate.yaml"

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project=str(YOLO_PROJECT),          # 👈 lưu run cùng thư mục với yolo.ipynb
    name="MTL_FOOD_PLATE_01",          # thư mục con của run
)

print("✅ Train xong.")
# KHÔNG dùng results.best nữa, tự build path
RUN_DIR = YOLO_PROJECT / "MTL_FOOD_PLATE_01"
YOLO_BEST = RUN_DIR / "weights" / "best.pt"
print("Best weights nằm ở:", YOLO_BEST)


NB_DIR: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo
New https://pypi.org/project/ultralytics/8.3.228 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (Quadro RTX 5000, 15925MiB)
engine/trainer: task=detect, mode=train, model=yolo11s.pt, data=/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/data_food_plate.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=/media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo, name=MTL_FOOD_PLATE_01, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=3

train: Scanning /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/labels/train.cache... 302 images, 11 backgrounds, 0 corrupt: 100%|██████████| 302/302 [00:00<?, ?it/s]
val: Scanning /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/labels/val.cache... 76 images, 3 backgrounds, 0 corrupt: 100%|██████████| 76/76 [00:00<?, ?it/s]


Plotting labels to /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo/MTL_FOOD_PLATE_01/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo/MTL_FOOD_PLATE_01
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      4.43G     0.7832      1.905      1.195        216        640: 100%|██████████| 19/19 [00:17<00:00,  1.08it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:06<00:00,  2.15s/it]

                   all         76        618      0.604      0.759       0.72      0.577


...

                   all         76        618      0.724      0.743      0.778      0.696



100 epochs completed in 0.124 hours.
Optimizer stripped from /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo/MTL_FOOD_PLATE_01/weights/last.pt, 19.2MB
Optimizer stripped from /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo/MTL_FOOD_PLATE_01/weights/best.pt, 19.2MB

Validating /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo/MTL_FOOD_PLATE_01/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (Quadro RTX 5000, 15925MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:00<00:00,  4.24it/s]


                   all         76        618      0.781      0.725      0.796      0.706
Speed: 0.3ms preprocess, 3.0ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo/MTL_FOOD_PLATE_01
✅ Train xong.


AttributeError: 'DetMetrics' object has no attribute 'best'. See valid attributes below.

    Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP) of an
    object detection model.

    Args:
        save_dir (Path): A path to the directory where the output plots will be saved. Defaults to current directory.
        plot (bool): A flag that indicates whether to plot precision-recall curves for each class. Defaults to False.
        on_plot (func): An optional callback to pass plots path and data when they are rendered. Defaults to None.
        names (dict of str): A dict of strings that represents the names of the classes. Defaults to an empty tuple.

    Attributes:
        save_dir (Path): A path to the directory where the output plots will be saved.
        plot (bool): A flag that indicates whether to plot the precision-recall curves for each class.
        on_plot (func): An optional callback to pass plots path and data when th

In [50]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Đường dẫn weight YOLO mới
YOLO_BEST = Path("MTL_FOOD_PLATE_01/weights/best.pt")

det_model = YOLO(str(YOLO_BEST))
print("Loaded:", YOLO_BEST)

# Chọn 1 ảnh bất kỳ trong val để test
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")
VAL_IMG_DIR = ROOT / "images" / "val"

val_imgs = sorted([p for p in VAL_IMG_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
print("Số ảnh val:", len(val_imgs))

test_img = val_imgs[0]
print("Test image:", test_img)

# Predict với YOLO mới
results = det_model.predict(
    source=str(test_img),
    conf=0.4,
    verbose=False,
)

# Ultralytics có hàm vẽ sẵn
res = results[0]
plot = res.plot()   # numpy array BGR

# show bằng matplotlib
img_show = Image.fromarray(plot[..., ::-1])  # BGR -> RGB
plt.figure(figsize=(8, 8))
plt.imshow(img_show)
plt.axis("off")
plt.show()


Loaded: MTL_FOOD_PLATE_01/weights/best.pt
Số ảnh val: 76
Test image: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/images/val/1669946305-com-44-16697116830802086116054-width2000height2024.jpeg


<Figure size 800x800 with 1 Axes>

In [52]:
from ultralytics import YOLO
from pathlib import Path

YOLO_WEIGHTS = Path("MTL_FOOD_PLATE_01/weights/best.pt")
yolo_model = YOLO(str(YOLO_WEIGHTS))
print("✅ Loaded plate detector:", YOLO_WEIGHTS)


✅ Loaded plate detector: MTL_FOOD_PLATE_01/weights/best.pt


In [53]:
from PIL import Image

def detect_food_boxes(image_path, conf_thres=0.4):
    """
    Dùng YOLO plate detector để tìm các dĩa/tô trên bàn.
    Trả về:
      - img (PIL.Image)
      - boxes: list (x1, y1, x2, y2, cls_id, score)
    """
    img = Image.open(image_path).convert("RGB")

    results = yolo_model.predict(
        source=str(image_path),
        conf=conf_thres,
        verbose=False,
    )

    boxes = []
    for r in results:
        h, w = r.orig_img.shape[:2]
        if r.boxes is None:
            continue

        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().tolist()
            score = float(b.conf.item())
            cls_id = int(b.cls.item())  # YOLO plate: luôn là 0 (dish)

            boxes.append((x1, y1, x2, y2, cls_id, score))

    print(f"YOLO phát hiện {len(boxes)} dĩa/tô (conf ≥ {conf_thres}).")
    return img, boxes


In [54]:
from pathlib import Path
import sys
import json

import torch
from torchvision import transforms

# Nếu chưa có ROOT_DIR thì chắc chắn lại:
# - Notebook trong Jupyter/ → ROOT_DIR = Path.cwd()
# - Notebook trong Jupyter/evaluate/ → ROOT_DIR = parent
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "Yolo":
    ROOT_DIR = NOTEBOOK_DIR.parent
else:
    ROOT_DIR = NOTEBOOK_DIR

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("ROOT_DIR    :", ROOT_DIR)

# Thêm models vào sys.path
sys.path.append(str(ROOT_DIR / "models"))

from efficientnet_b0 import (
    build_model,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ====== CHỌN CHECKPOINT CẦN DÙNG ======
# 👉 chỉnh đúng run & tên file ở đây
RUN_NAME = "MTL_TGFOOD"          # ví dụ: run incremental 34 lớp
CKPT_FILE = "mtl_effcientnet_b0_best.pt"                  # hoặc "mtl_effb0_best.pt" với model 33 lớp

CKPT_PATH = ROOT_DIR / "runs" / RUN_NAME / "checkpoints" / CKPT_FILE
RUN_DIR   = CKPT_PATH.parent.parent

print("Classifier ckpt:", CKPT_PATH)
print("Run dir        :", RUN_DIR)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {CKPT_PATH}")

# ====== LOAD CHECKPOINT ======
ckpt = torch.load(CKPT_PATH, map_location=device)

# 1) ưu tiên class_names trong checkpoint
class_names = ckpt.get("class_names", None)

# 2) nếu không có thì thử đọc classes.txt trong run
if class_names is None:
    classes_txt = RUN_DIR / "classes.txt"
    if classes_txt.exists():
        print("📄 Đọc class_names từ:", classes_txt)
        lines = classes_txt.read_text(encoding="utf-8").splitlines()
        class_names = [ln.strip() for ln in lines if ln.strip()]

# 3) nếu vẫn chưa có thì fallback sang runs_meta/class_names.json (global)
if class_names is None:
    runs_meta = ROOT_DIR.parent / "runs_meta" / "class_names.json"
    print("⚠️ Không tìm thấy class_names trong ckpt/run → fallback:", runs_meta)
    with open(runs_meta, "r", encoding="utf-8") as f:
        class_names = json.load(f)

num_classes = len(class_names)
print("Số class:", num_classes)
print("Một vài class đầu:", class_names[:10])

# ====== BUILD MODEL ĐÚNG SỐ LỚP & LOAD WEIGHTS ======
model = build_model(
    num_classes=num_classes,
    dropout=0.4,
    pretrained=False,      # dùng weight từ ckpt nên không cần ImageNet
    freeze_backbone=False, # chỉ để build, inference không ảnh hưởng
    device=device,
)

# Nếu checkpoint lưu kiểu {'model_state': ..., ...}
state = ckpt.get("model_state", ckpt)
model.load_state_dict(state)
model.eval()

print("✅ Loaded classifier model.")

# ====== TRANSFORM EVAL (KHÔNG AUGMENT) ======
to_rgb = transforms.Lambda(lambda im: im.convert("RGB"))
eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    to_rgb,
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


NOTEBOOK_DIR: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo
ROOT_DIR    : /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter
Device: cuda
Classifier ckpt: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/runs/MTL_TGFOOD/checkpoints/mtl_effcientnet_b0_best.pt
Run dir        : /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/runs/MTL_TGFOOD
Số class: 33
Một vài class đầu: ['Banh beo', 'Banh bot loc', 'Banh can', 'Banh canh', 'Banh chung', 'Banh cuon', 'Banh duc', 'Banh gio', 'Banh khot', 'Banh mi']
✅ torch.compile enabled
✅ Loaded classifier model.


In [55]:
import torch
from torchvision import transforms

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval().to(device)

# transform infer = y chang eval_tfms trong efficientnet_b0.py
infer_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.no_grad()
def crop_and_classify(img, boxes, prob_thres=0.5, expand_scale=1.2):
    """
    img   : PIL Image gốc (bàn ăn)
    boxes : list (x1, y1, x2, y2, cls_id, score) từ YOLO
    """
    w, h = img.size
    results = []
    counts = {}

    for (x1, y1, x2, y2, _, det_conf) in boxes:
        # nới box một chút quanh dĩa
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bw = (x2 - x1) * expand_scale
        bh = (y2 - y1) * expand_scale

        nx1 = max(0, cx - bw / 2)
        ny1 = max(0, cy - bh / 2)
        nx2 = min(w, cx + bw / 2)
        ny2 = min(h, cy + bh / 2)

        crop = img.crop((nx1, ny1, nx2, ny2))
        tensor = infer_tfms(crop).unsqueeze(0).to(device)

        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_prob, pred_idx = torch.max(probs, dim=0)

        pred_prob = float(pred_prob.item())
        pred_idx = int(pred_idx.item())
        pred_name = class_names[pred_idx]   # list class_names load từ runs_meta

        if pred_prob < prob_thres:
            continue  # bỏ những box classifier không chắc

        results.append({
            "box": (nx1, ny1, nx2, ny2),
            "det_conf": float(det_conf),
            "pred_idx": pred_idx,
            "pred_name": pred_name,
            "pred_prob": pred_prob,
        })

        counts[pred_name] = counts.get(pred_name, 0) + 1

    return results, counts


In [58]:
from PIL import ImageDraw, ImageFont
import matplotlib.pyplot as plt

def visualize_results(img, results, font_size=16, save_path=None):
    drawn = img.copy()
    draw = ImageDraw.Draw(drawn)

    # font: cố gắng dùng font hệ thống, nếu fail thì dùng default
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    for r in results:
        x1, y1, x2, y2 = r["bbox"]
        dish = r["dish_name"]
        prob = r["prob"]

        label = f"{dish} ({prob:.2f})"
        draw.rectangle((x1, y1, x2, y2), outline="red", width=2)

        # ===== đo kích thước chữ cho Pillow mới =====
        if hasattr(draw, "textbbox"):
            bbox = draw.textbbox((0, 0), label, font=font)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
        else:
            tw, th = draw.textsize(label, font=font)
        # ============================================

        # nền đen cho label
        draw.rectangle((x1, y1 - th - 4, x1 + tw + 4, y1), fill="black")
        draw.text((x1 + 2, y1 - th - 2), label, fill="yellow", font=font)

    # nếu có đường dẫn lưu thì save ra file
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        drawn.save(save_path)
        print(f"💾 Đã lưu ảnh có box tại: {save_path}")

    # vẫn show trong notebook
    plt.figure(figsize=(8, 8))
    plt.imshow(drawn)
    plt.axis("off")
    plt.show()


In [59]:
def print_summary_counts(counts):
    """
    counts: dict {class_name: count}
    In tổng kết số món trên bàn theo class.
    """
    if not counts:
        print("👉 Không có món nào được classifier nhận ra (sau khi lọc prob).")
        return

    total = sum(counts.values())
    print("👉 Bàn ăn có tổng cộng:", total, "món (theo classifier).")
    print()
    for name, c in sorted(counts.items(), key=lambda x: x[0]):
        print(f"  - {name}: {c}")


In [60]:
import matplotlib.pyplot as plt

def debug_show_crops(img, results, max_show=10):
    """
    img      : PIL image gốc
    results  : list các kết quả từ crop_and_classify
               (dict hoặc tuple đều được)
    """

    def get_box(r):
        # r là dict
        if isinstance(r, dict):
            if "box" in r:           # kiểu {'box': (x1,y1,x2,y2), ...}
                return r["box"]
            if "bbox" in r:          # kiểu {'bbox': [x1,y1,x2,y2], ...}
                return r["bbox"]
            if all(k in r for k in ("x1", "y1", "x2", "x2")):
                return (r["x1"], r["y1"], r["x2"], r["y2"])
        # r là list / tuple: (x1,y1,x2,y2, ...)
        x1, y1, x2, y2 = r[:4]
        return (x1, y1, x2, y2)

    def get_name_prob(r):
        if isinstance(r, dict):
            name = r.get("pred_name") or r.get("name") or r.get("cls_name") or "?"
            prob = r.get("pred_prob") or r.get("prob") or r.get("score") or 0.0
        else:
            # nếu là tuple thì tạm lấy name ở vị trí 4, prob ở vị trí 5 (nếu có)
            name = "?"
            prob = 0.0
            if len(r) >= 5:
                name = str(r[4])
            if len(r) >= 6:
                prob = float(r[5])
        return name, prob

    n = min(len(results), max_show)
    if n == 0:
        print("Không có kết quả nào trong results để debug.")
        return

    plt.figure(figsize=(4 * n, 4))

    for i, r in enumerate(results[:n], start=1):
        x1, y1, x2, y2 = get_box(r)
        name, prob = get_name_prob(r)

        crop = img.crop((x1, y1, x2, y2))

        ax = plt.subplot(1, n, i)
        ax.imshow(crop)
        ax.axis("off")
        ax.set_title(f"{name}\n({prob:.2f})")

    plt.tight_layout()
    plt.show()


In [66]:
from pathlib import Path

# 1) Chọn ảnh thật (bàn ăn ngoài đời)

test_image = Path("/home/mtl/Downloads/6.jpg")  # ✅ Path
# hoặc 1 ảnh bất kỳ ông thích, free thay đường dẫn

print("Test image:", test_image)

# 2) YOLO detect các dĩa/tô
img, boxes = detect_food_boxes(str(test_image), conf_thres=0.4)

# 3) EfficientNet crop + classify
results, counts = crop_and_classify(
    img,
    boxes,
    prob_thres=0.8,     # bỏ bớt box classifier không chắc
    expand_scale=0.8,   # nới box rộng hơn dĩa
)

# 4) Lưu ảnh có box ra thư mục output
out_dir = ROOT_DIR / "images" / "detect_foods"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"{test_image.stem}_det{test_image.suffix}"

#visualize_results(img, results, save_path=out_path)
debug_show_crops(img, results, max_show=12)

# 5) In tổng kết số món
print_summary_counts(counts)
print("💾 Đã lưu ảnh có box tại:", out_path)


Test image: /home/mtl/Downloads/6.jpg
YOLO phát hiện 23 dĩa/tô (conf ≥ 0.4).


<Figure size 4800x400 with 12 Axes>

👉 Bàn ăn có tổng cộng: 19 món (theo classifier).

  - Banh beo: 1
  - Banh duc: 1
  - Banh trang nuong: 1
  - Banh xeo: 1
  - Canh chua: 2
  - Nem chua: 2
  - Xoi xeo: 5
  - banh_tieu: 6
💾 Đã lưu ảnh có box tại: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/images/detect_foods/6_det.jpg
